# 04. Снепшот поточних курсів

Цей ноутбук відповідає за четвертий етап конвеєра. Він читає дані з bronze та створює таблицю з актуальним курсом кожної валюти.

Снепшот показує стан даних на момент його побудови. Для кожної валюти в ньому повинен залишитися лише один найсвіжіший рядок.

Цей ноутбук не звертається до API НБУ. Він використовує дані з таблиці `nbu_raw.raw_rates` та ключі валют із таблиці `nbu_dwh.dim_currency`.

Результат записується в режимі `WRITE_TRUNCATE`. Під час кожного запуску попередній снепшот повністю замінюється новим.

## Послідовність роботи

1. Підключитися до BigQuery.
2. Прочитати дані з bronze.
3. Розгорнути JSON і нормалізувати значення.
4. Залишити найсвіжіший курс кожної валюти.
5. Приєднати `currency_key` із виміру валют.
6. Додати момент побудови снепшоту.
7. Записати результат у `nbu_dwh.snapshot_rates_current`.
8. Перевірити отриману таблицю.

## 1. Налаштування та підключення

Вказуємо назву Google Cloud проєкту, імпортуємо потрібні бібліотеки та створюємо клієнт BigQuery.

Ноутбук підключається до BigQuery самостійно, оскільки оркестратор запускатиме кожен етап окремо.

Для авторизації використовуємо JSON-ключ сервісного акаунта зі змінної середовища `GCP_SA_KEY`. Сам ключ у коді не зберігається.

In [1]:
PROJECT_ID = "nbu-bigquery-etl"
LOCATION = "EU"

DS_RAW = "nbu_raw"
DS_DWH = "nbu_dwh"

RAW_TABLE = f"{PROJECT_ID}.{DS_RAW}.raw_rates"
DIM_CURRENCY_TABLE = f"{PROJECT_ID}.{DS_DWH}.dim_currency"
SNAPSHOT_TABLE = f"{PROJECT_ID}.{DS_DWH}.snapshot_rates_current"

In [2]:
import os
import sys
import json

import pandas as pd
from google.cloud import bigquery

pd.set_option("display.max_columns", 40)

In [3]:
creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account

    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]

    )

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)

print("проєкт:", client.project)

проєкт: nbu-bigquery-etl


## Завдання 5. Ноутбук 04: снепшот поточних курсів

### Завдання 5.1. Читання та підготовка даних із bronze

Читаємо з таблиці `nbu_raw.raw_rates` момент завантаження, дату курсу та JSON-текст із даними валюти.

Розгортаємо `payload` в окремі колонки та нормалізуємо значення так само, як у ноутбуці з виміром валют. Код валюти приводимо до верхнього регістру, назву очищуємо від зайвих пробілів, а числові поля переводимо до відповідних числових типів.

Колонка `business_date` потрібна для вибору найсвіжішої дати курсу. Колонка `ingested_at` допоможе вибрати найсвіжіше завантаження, якщо за одну дату в bronze є кілька однакових записів.

In [4]:
sql = f"""
SELECT ingested_at,
       business_date,
       payload
  FROM `{RAW_TABLE}`
"""

raw = client.query(sql).to_dataframe()

print(len(raw), "рядків прочитано з bronze")
raw.head(5)

90 рядків прочитано з bronze


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ingested_at,business_date,payload
0,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""DZD"", ""exchangedate"": ""24.08.2026"", ""r..."
1,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""AUD"", ""exchangedate"": ""24.08.2026"", ""r..."
2,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""BDT"", ""exchangedate"": ""24.08.2026"", ""r..."
3,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""CAD"", ""exchangedate"": ""24.08.2026"", ""r..."
4,2026-08-24 08:14:35+00:00,2026-08-24,"{""cc"": ""CNY"", ""exchangedate"": ""24.08.2026"", ""r..."


In [5]:
parsed = pd.json_normalize(raw["payload"].map(json.loads))

print("колонки JSON:", list(parsed.columns))
print(len(parsed), "рядків після розгортання JSON")

parsed.head(5)

колонки JSON: ['cc', 'exchangedate', 'r030', 'rate', 'special', 'txt']
90 рядків після розгортання JSON


,cc,exchangedate,r030,rate,special,txt
0,DZD,24.08.2026,12,0.33613,NaN,Алжирський динар
1,AUD,24.08.2026,36,31.99060,NaN,Австралійський долар
2,BDT,24.08.2026,50,0.36509,NaN,Така
3,CAD,24.08.2026,124,32.50720,NaN,Канадський долар
4,CNY,24.08.2026,156,6.64470,NaN,Юань Женьміньбі


In [6]:
parsed["currency_code"] = parsed["cc"].str.strip().str.upper()
parsed["currency_name"] = parsed["txt"].str.strip()
parsed["r030"] = pd.to_numeric(parsed["r030"], errors='coerce').astype('Int64')
parsed["rate"] = pd.to_numeric(parsed["rate"], errors="coerce")

currency_data = parsed[["currency_code", "currency_name", "r030", "rate"]].copy()

df = pd.concat(
    [
        raw[["ingested_at", "business_date"]].reset_index(drop=True),
        currency_data.reset_index(drop=True)
    ],
    axis=1
)

print("колонки:", list(df.columns))
print(len(df), "рядків після нормалізації")

df.head(5)

колонки: ['ingested_at', 'business_date', 'currency_code', 'currency_name', 'r030', 'rate']
90 рядків після нормалізації


,ingested_at,business_date,currency_code,currency_name,r030,rate
0,2026-08-24 08:14:35+00:00,2026-08-24,DZD,Алжирський динар,12,0.33613
1,2026-08-24 08:14:35+00:00,2026-08-24,AUD,Австралійський долар,36,31.99060
2,2026-08-24 08:14:35+00:00,2026-08-24,BDT,Така,50,0.36509
3,2026-08-24 08:14:35+00:00,2026-08-24,CAD,Канадський долар,124,32.50720
4,2026-08-24 08:14:35+00:00,2026-08-24,CNY,Юань Женьміньбі,156,6.64470


### Завдання 5.2. Вибір найсвіжішого курсу кожної валюти

У bronze одна валюта може повторюватися за різні дати та після повторних завантажень.

Сортуємо дані спочатку за `business_date`, а потім за `ingested_at`. Для кожної валюти залишаємо останній рядок. Таким способом вибираємо курс за найсвіжішу дату та найсвіжіше завантаження цієї дати.

Після цього в наборі повинен залишитися один рядок на кожен `currency_code`.

In [7]:
snap_latest  = (df.sort_values(["business_date", "ingested_at"])
                .drop_duplicates(subset=["currency_code"], keep="last")).reset_index(drop=True)

print(len(df), "рядків до вибору актуальних курсів")
print(len(snap_latest), "актуальних валют")

snap_latest.head(5)

90 рядків до вибору актуальних курсів
45 актуальних валют


,ingested_at,business_date,currency_code,currency_name,r030,rate
0,2026-08-24 08:15:17+00:00,2026-08-24,DZD,Алжирський динар,12,0.33613
1,2026-08-24 08:15:17+00:00,2026-08-24,AUD,Австралійський долар,36,31.99060
2,2026-08-24 08:15:17+00:00,2026-08-24,BDT,Така,50,0.36509
3,2026-08-24 08:15:17+00:00,2026-08-24,CAD,Канадський долар,124,32.50720
4,2026-08-24 08:15:17+00:00,2026-08-24,CNY,Юань Женьміньбі,156,6.64470


### Завдання 5.3. Приєднання ключів валют

Читаємо з таблиці `nbu_dwh.dim_currency` код валюти та її внутрішній ключ.

Приєднуємо `currency_key` до актуальних курсів за `currency_code`. Ключі не створюємо повторно, оскільки всі таблиці сховища повинні використовувати один спільний довідник валют.

Якщо код валюти не буде знайдений у вимірі, підставляємо технічний ключ `-1`.

In [8]:
sql = f"""
SELECT currency_key,
       currency_code
  FROM `{DIM_CURRENCY_TABLE}`
 ORDER BY currency_key
"""

dim_currency = client.query(sql).to_dataframe()

print(len(dim_currency), f"рядків прочитано з {DIM_CURRENCY_TABLE}")
dim_currency.head(5)

46 рядків прочитано з nbu-bigquery-etl.nbu_dwh.dim_currency


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,currency_key,currency_code
0,-1,N/A
1,1,AED
2,2,AUD
3,3,AZN
4,4,BDT


In [9]:
snap = snap_latest.merge(dim_currency, how="left", on="currency_code", validate="one_to_one")

snap["currency_key"] = (snap["currency_key"].fillna(-1).astype(int))

print("рядків після об’єднання:", len(snap))
print("валют із currency_key = -1:", (snap["currency_key"] == -1).sum())

snap.head(5)

рядків після об’єднання: 45
валют із currency_key = -1: 0


,ingested_at,business_date,currency_code,currency_name,r030,rate,currency_key
0,2026-08-24 08:15:17+00:00,2026-08-24,DZD,Алжирський динар,12,0.33613,10
1,2026-08-24 08:15:17+00:00,2026-08-24,AUD,Австралійський долар,36,31.99060,2
2,2026-08-24 08:15:17+00:00,2026-08-24,BDT,Така,50,0.36509,4
3,2026-08-24 08:15:17+00:00,2026-08-24,CAD,Канадський долар,124,32.50720,5
4,2026-08-24 08:15:17+00:00,2026-08-24,CNY,Юань Женьміньбі,156,6.64470,7


### Завдання 5.4. Додавання часу побудови снепшоту

Додаємо колонку `snapshot_ts` із поточним часом у часовій зоні UTC. Вона показує, коли саме був побудований снепшот.

Усі рядки отримують однаковий час, оскільки належать до одного запуску ноутбука.

Після цього залишаємо тільки фінальні колонки та сортуємо результат за `currency_key`.

In [10]:
snap["snapshot_ts"] = pd.Timestamp.now(tz="UTC").floor("s")

snap = snap[["currency_key", "currency_code", "currency_name", "rate",
             "business_date", "snapshot_ts"]].sort_values(["currency_key"]).reset_index(drop=True)

snap.head(5)

,currency_key,currency_code,currency_name,rate,business_date,snapshot_ts
0,1,AED,Дирхам ОАЕ,12.15870,2026-08-24,2026-08-24 08:16:44+00:00
1,2,AUD,Австралійський долар,31.99060,2026-08-24,2026-08-24 08:16:44+00:00
2,3,AZN,Азербайджанський манат,26.27150,2026-08-24,2026-08-24 08:16:44+00:00
3,4,BDT,Така,0.36509,2026-08-24,2026-08-24 08:16:44+00:00
4,5,CAD,Канадський долар,32.50720,2026-08-24,2026-08-24 08:16:44+00:00


### Завдання 5.5. Запис снепшоту в BigQuery

Записуємо підготовлений DataFrame у таблицю `nbu_dwh.snapshot_rates_current`.

Схему задаємо явно, щоб BigQuery отримав потрібні назви й типи колонок. Для запису використовуємо режим `WRITE_TRUNCATE`, тому кожен запуск повністю замінює попередній снепшот.

Старі версії тут не зберігаються, оскільки ця таблиця показує тільки актуальний стан курсів.

In [11]:
snap_schema = [
    bigquery.SchemaField("currency_key",  "INTEGER",   mode="REQUIRED"),
    bigquery.SchemaField("currency_code", "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("currency_name", "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("rate",          "FLOAT",     mode="REQUIRED"),
    bigquery.SchemaField("business_date", "DATE",      mode="REQUIRED"),
    bigquery.SchemaField("snapshot_ts",   "TIMESTAMP", mode="REQUIRED"),
]

cfg = bigquery.LoadJobConfig(schema=snap_schema, write_disposition="WRITE_TRUNCATE")

load_job = client.load_table_from_dataframe(snap, SNAPSHOT_TABLE, job_config=cfg)

load_job.result()

table = client.get_table(SNAPSHOT_TABLE)

print("таблиця:", SNAPSHOT_TABLE)
print("записано рядків:", table.num_rows)

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


таблиця: nbu-bigquery-etl.nbu_dwh.snapshot_rates_current
записано рядків: 45


### Завдання 5.6. Перевірка снепшоту

Читаємо записаний снепшот із BigQuery та перевіряємо три умови.

Кількість рядків повинна дорівнювати кількості унікальних валют у bronze. Кожен `currency_code` повинен зустрічатися лише один раз. Також усі валюти повинні мати правильний `currency_key`, без використання технічного ключа `-1`.

In [12]:
sql = f"""
SELECT currency_key,
       currency_code,
       currency_name,
       rate,
       business_date,
       snapshot_ts
  FROM `{SNAPSHOT_TABLE}`
ORDER BY currency_key
"""

check_snap = client.query(sql).to_dataframe()

print(len(check_snap), f"рядків прочитано з {SNAPSHOT_TABLE}")
check_snap.head(5)

45 рядків прочитано з nbu-bigquery-etl.nbu_dwh.snapshot_rates_current


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,currency_key,currency_code,currency_name,rate,business_date,snapshot_ts
0,1,AED,Дирхам ОАЕ,12.15870,2026-08-24,2026-08-24 08:16:44+00:00
1,2,AUD,Австралійський долар,31.99060,2026-08-24,2026-08-24 08:16:44+00:00
2,3,AZN,Азербайджанський манат,26.27150,2026-08-24,2026-08-24 08:16:44+00:00
3,4,BDT,Така,0.36509,2026-08-24,2026-08-24 08:16:44+00:00
4,5,CAD,Канадський долар,32.50720,2026-08-24,2026-08-24 08:16:44+00:00


In [13]:
print("кількість рядків правильна:", len(check_snap) == df["currency_code"].nunique())

print("currency_code унікальний:", check_snap["currency_code"].is_unique)

print("немає currency_key = -1:", not (check_snap["currency_key"] == -1).any())

кількість рядків правильна: True
currency_code унікальний: True
немає currency_key = -1: True


In [14]:
print("04_snapshot завершено успішно")

04_snapshot завершено успішно
